In [2]:
import torch
x = torch.zeros(4,8)

In [3]:
x.numel()

32

In [4]:
a = torch.tensor([1e-8], dtype = torch.float16)
assert a == 0

In [5]:
a = torch.tensor([1e-8], dtype = torch.bfloat16)
assert a != 0

In [6]:
torch.finfo(torch.float32)

finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)

In [7]:
torch.finfo(torch.float16)

finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)

In [8]:
torch.finfo(torch.bfloat16)

finfo(resolution=0.01, min=-3.38953e+38, max=3.38953e+38, eps=0.0078125, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=bfloat16)

In [9]:
from einops import *

x = torch.ones(3,4)
y = torch.ones(4,3)

# Old way
z = x @ y

In [10]:
# New way

z_einops = einsum(x, y, "seq1 hidden, hidden seq2 -> seq1 seq2")

In [11]:
z == z_einops

tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])

In [12]:
x = torch.ones(2,3,4)
y = torch.ones(2,3,4)

In [13]:
z_old = x @ y.transpose(-2,-1)

In [14]:
z_ei = einsum(x,y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2" )

In [15]:
x = torch.ones(3,8)
x = rearrange(x, "... (heads hidden1) -> ... heads hidden1", heads = 2)
x.shape

torch.Size([3, 2, 4])

In [21]:
B = 1024
D = 256
K = 64
device = "cuda"
x = torch.ones(B,D, device=device)
w = torch.ones(D,K, device= device)

In [50]:
y = x @ w

How Many flops? 
A single scalar product takes D mul and D-1 sum, suppose D 2 * D, then I have to do it for every columns and for every row, so total flops = 2 * D * B* K

In [102]:
import timeit

B = 4096
D = 4096
K = 4096
actual_num_flops = 2 * D * B * K
x = torch.ones(B,D, device=device,dtype=torch.bfloat16)
w = torch.ones(D,K, device= device,dtype=torch.bfloat16)
def run():
   x @ w 
   torch.cuda.synchronize()
num_trials = int(10)
total_time = timeit.timeit(run, number = num_trials)
actual_time = total_time/num_trials
actual_flop_per_sec = actual_num_flops / actual_time
actual_flop_per_sec

3960378379571.6504

In [61]:
actual_num_flops = 2 * D * B * K

In [62]:
actual_flop_per_sec = actual_num_flops / actual_time

In [63]:
actual_flop_per_sec

273797416893.00964